# 🎵 Spotify Hit Prediction — Production ML Pipeline
### End-to-End: EDA → Feature Engineering → Modeling → XAI → Explainability → Export

| Phase | Description |
|-------|-------------|
| **0** | Environment Setup & GPU Detection |
| **1** | Data Loading (auto-detects 1 or 2 CSVs) |
| **2** | Advanced EDA with Statistical Tests |
| **3** | Advanced Feature Engineering |
| **4** | Baseline Model Framework (13 classifiers) |
| **5** | Advanced Tuning via Optuna + GPU Acceleration |
| **6** | Ensemble & Stacking |
| **7** | Full Model Comparison with Visuals |
| **8** | XAI: SHAP, Permutation Importance, Waterfall |
| **9** | Save All Outputs + ZIP Download |

> **Dataset:** Upload `high_popularity_spotify_data.csv` + `low_popularity_spotify_data.csv` when prompted.  
> **Runtime:** GPU preferred (T4 / A100). CPU also supported.

## ⚙️ Phase 0 — Environment Setup

In [1]:
# ─────────────────────────────────────────────────────────────
# Phase 0 | Environment Setup & Library Installation
# ─────────────────────────────────────────────────────────────
import subprocess, sys

def _install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_install(
    "shap",
    "xgboost",
    "lightgbm",
    "catboost",
    "imbalanced-learn",
    "optuna",
    "plotly",
    "kaleido"
)

import warnings
warnings.filterwarnings("ignore")

# ── Standard ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, mannwhitneyu

# ── Sklearn ─────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier,
    BaggingClassifier, VotingClassifier, StackingClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve,
    ConfusionMatrixDisplay, matthews_corrcoef, log_loss, brier_score_loss
)
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# ── Boosting ────────────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# ── Tuning ──────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── XAI ─────────────────────────────────────────────────────
import shap

# ── Utils ───────────────────────────────────────────────────
import os, zipfile, time, json, pickle
from datetime import datetime
from pathlib import Path
from collections import defaultdict

# ── Globals ─────────────────────────────────────────────────
SEED       = 42
np.random.seed(SEED)
TIMESTAMP  = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIR = Path("spotify_outputs")
for sub in ["eda", "features", "models", "xai", "comparison", "reports"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ── Plot theme ───────────────────────────────────────────────
PALETTE    = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6",
              "#1abc9c", "#e67e22", "#2980b9", "#8e44ad", "#27ae60"]
plt.rcParams.update({
    "figure.dpi"       : 130,
    "font.size"        : 11,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.titlesize"   : 13,
    "axes.labelsize"   : 11,
})
sns.set_palette(PALETTE)

print(f"✅  Environment ready — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁  Outputs → {OUTPUT_DIR.resolve()}")

✅  Environment ready — 2026-06-08 04:26:31
📁  Outputs → /content/spotify_outputs


## 📂 Phase 1 — Data Loading & Validation

In [2]:
# ─────────────────────────────────────────────────────────────
# Phase 1 | Data Loading
# Accepts:
#   (a) high_popularity_spotify_data.csv + low_popularity_spotify_data.csv
#   (b) Any single CSV with a 'track_popularity' column
# ─────────────────────────────────────────────────────────────
from google.colab import files as _colab_files

print("📂  Upload your Spotify CSV file(s):")
print("    → high_popularity_spotify_data.csv  (popularity 68–100)")
print("    → low_popularity_spotify_data.csv   (popularity 11–68)")
print("    or any single merged CSV\n")

uploaded      = _colab_files.upload()
upload_names  = list(uploaded.keys())
print(f"\nUploaded: {upload_names}")

# ── Auto-detect layout ──────────────────────────────────────
_high = [f for f in upload_names if "high" in f.lower()]
_low  = [f for f in upload_names if "low"  in f.lower()]
_other= [f for f in upload_names if f not in _high + _low]

if _high and _low:
    df_high = pd.read_csv(_high[0])
    df_low  = pd.read_csv(_low[0])
    # Normalise column name differences (columns are same order but may differ)
    for _df in [df_high, df_low]:
        for cand in ["track_popularity", "popularity"]:
            if cand in _df.columns:
                _df.rename(columns={cand: "track_popularity"}, inplace=True)
                break
    df_high["Popularity_Label"] = "High"
    df_low["Popularity_Label"]  = "Low"
    df_raw = pd.concat([df_high, df_low], ignore_index=True)
    print(f"\n✅  Merged → High({len(df_high):,}) + Low({len(df_low):,}) = {len(df_raw):,} rows")
elif _other:
    df_raw = pd.read_csv(_other[0])
    for cand in ["track_popularity", "popularity"]:
        if cand in df_raw.columns:
            df_raw.rename(columns={cand: "track_popularity"}, inplace=True)
            break
    median_pop = df_raw["track_popularity"].median()
    df_raw["Popularity_Label"] = (df_raw["track_popularity"]
                                   .apply(lambda x: "High" if x >= median_pop else "Low"))
    print(f"\n✅  Single file → {len(df_raw):,} rows, median split at {median_pop}")
else:
    raise FileNotFoundError("No CSV recognised. Please re-upload.")

# ── Quick sanity ────────────────────────────────────────────
assert "track_popularity" in df_raw.columns, "Column 'track_popularity' not found!"
print(f"Shape         : {df_raw.shape}")
print(f"Pop range     : {df_raw['track_popularity'].min()} – {df_raw['track_popularity'].max()}")
print(f"Label balance : {df_raw['Popularity_Label'].value_counts().to_dict()}")
print(f"\nFirst 3 rows:")
display(df_raw.head(3))

📂  Upload your Spotify CSV file(s):
    → high_popularity_spotify_data.csv  (popularity 68–100)
    → low_popularity_spotify_data.csv   (popularity 11–68)
    or any single merged CSV



Saving high_popularity_spotify_data.csv to high_popularity_spotify_data (1).csv
Saving low_popularity_spotify_data.csv to low_popularity_spotify_data (1).csv

Uploaded: ['high_popularity_spotify_data (1).csv', 'low_popularity_spotify_data (1).csv']

✅  Merged → High(1,686) + Low(3,145) = 4,831 rows
Shape         : (4831, 30)
Pop range     : 11 – 100
Label balance : {'Low': 3145, 'High': 1686}

First 3 rows:


,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id,Popularity_Label
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3.0,0.0304,...,10FLjwfpbxLmW8c25Xyc2N,0.0,6.0,251668.0,0.308,2plbrEY59IikOBgBGLjaoe,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,High
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,4.0,0.0358,...,7aJuG4TFXa2hmE4z1yxc3n,1.0,2.0,210373.0,0.200,6dOtVTDdiauQNBQEDOtlAB,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,High
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,4.0,0.0368,...,0hBRqPYPXhr1RkTDG3n4Mk,1.0,1.0,166300.0,0.214,7ne4VBA60CxGM75vw0EYad,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,High


## 🔍 Phase 2 — Advanced EDA

In [3]:
# ─────────────────────────────────────────────────────────────
# Phase 2.1 | Data Quality Audit
# ─────────────────────────────────────────────────────────────
df = df_raw.copy()

AUDIO_FEATURES = [c for c in [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_ms",
    "time_signature", "key", "mode"
] if c in df.columns]

DROP_COLS = [c for c in [
    "track_href", "uri", "analysis_url", "type", "id"
] if c in df.columns]
df.drop(columns=DROP_COLS, inplace=True)

print("=" * 60)
print(f"  Shape           : {df.shape}")
print(f"  Audio features  : {len(AUDIO_FEATURES)}")
print(f"  Popularity range: {df['track_popularity'].min()} – {df['track_popularity'].max()}")
print("=" * 60)

# Missing values
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_df = pd.DataFrame({"Missing": miss, "Pct%": miss_pct})
miss_df = miss_df[miss_df["Missing"] > 0]
if len(miss_df):
    print("\n⚠️  Missing values:")
    display(miss_df)
else:
    print("\n✅  No missing values detected.")

# ── Duplicates ─────────────────────────────────────────────
dups = df.duplicated(subset=["track_id"] if "track_id" in df.columns else None).sum()
print(f"\n🔁  Duplicate rows: {dups}")
if dups:
    df = df.drop_duplicates(
        subset=["track_id"] if "track_id" in df.columns else None
    ).reset_index(drop=True)
    print(f"   Removed duplicates → {len(df):,} rows remain")

print("\n📊  Summary Statistics:")
display(df[AUDIO_FEATURES + ["track_popularity"]].describe().round(3).T)

  Shape           : (4831, 25)
  Audio features  : 13
  Popularity range: 11 – 100

⚠️  Missing values:


,Missing,Pct%
energy,1,0.02
tempo,1,0.02
danceability,1,0.02
loudness,1,0.02
liveness,1,0.02
valence,1,0.02
time_signature,1,0.02
speechiness,1,0.02
track_album_name,1,0.02
instrumentalness,1,0.02



🔁  Duplicate rows: 336
   Removed duplicates → 4,495 rows remain

📊  Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
danceability,4494.0,0.620,0.189,0.059,0.522,0.653,0.757,0.979
energy,4494.0,0.579,0.250,0.000,0.424,0.627,0.773,0.998
loudness,4494.0,-9.487,7.287,-48.069,-10.606,-7.301,-5.390,1.318
speechiness,4494.0,0.100,0.101,0.022,0.038,0.055,0.115,0.927
acousticness,4494.0,0.351,0.329,0.000,0.057,0.238,0.615,0.996
instrumentalness,4494.0,0.212,0.359,0.000,0.000,0.000,0.300,0.991
liveness,4494.0,0.168,0.125,0.021,0.096,0.117,0.195,0.979
valence,4494.0,0.479,0.260,0.030,0.267,0.480,0.689,0.987
tempo,4494.0,118.275,28.706,48.232,96.029,118.082,137.085,241.426
duration_ms,4494.0,205575.976,82768.499,35375.000,158215.250,194041.500,232919.750,1355260.000


In [4]:
# ─────────────────────────────────────────────────────────────
# Phase 2.2 | Popularity Distribution
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Histogram with KDE
axes[0].hist(df["track_popularity"], bins=40,
             color="#3498db", edgecolor="white", alpha=0.85)
axes[0].axvline(df["track_popularity"].median(), color="#e74c3c",
                linestyle="--", linewidth=2,
                label=f"Median = {df['track_popularity'].median():.0f}")
axes[0].axvline(df["track_popularity"].mean(), color="#f39c12",
                linestyle="-.", linewidth=2,
                label=f"Mean = {df['track_popularity'].mean():.1f}")
axes[0].set_title("Track Popularity Distribution")
axes[0].set_xlabel("Popularity Score (0–100)")
axes[0].set_ylabel("Count")
axes[0].legend()

# KDE: High vs Low
for lbl, col in zip(["High", "Low"], ["#2ecc71", "#e74c3c"]):
    sub = df[df["Popularity_Label"] == lbl]["track_popularity"]
    sub.plot.kde(ax=axes[1], label=lbl, color=col, linewidth=2.5)
axes[1].set_title("Popularity KDE — High vs Low")
axes[1].set_xlabel("Popularity Score")
axes[1].legend()
axes[1].fill_between(
    *zip(*[(x, 0) for x in np.linspace(0, 100, 200)]), alpha=0.0
)

# Popularity by Genre
if "playlist_genre" in df.columns:
    gp = (df.groupby("playlist_genre")["track_popularity"]
            .agg(["mean", "std"])
            .sort_values("mean", ascending=True))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(gp)))
    axes[2].barh(gp.index, gp["mean"], xerr=gp["std"],
                 color=colors, capsize=4, alpha=0.9)
    axes[2].set_title("Avg Popularity by Genre (±1σ)")
    axes[2].set_xlabel("Mean Popularity")
else:
    axes[2].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda/01_popularity_distribution.png", bbox_inches="tight")
plt.show()
print("💡 Key Insight: Popularity is bimodal — tracks are either viral hits or obscure.")
print("   Genre context significantly shifts the baseline — pop/edm genres score higher.")

💡 Key Insight: Popularity is bimodal — tracks are either viral hits or obscure.
   Genre context significantly shifts the baseline — pop/edm genres score higher.


In [5]:
# ─────────────────────────────────────────────────────────────
# Phase 2.3 | Audio Feature Distributions (High vs Low)
# ─────────────────────────────────────────────────────────────
n_feats = len(AUDIO_FEATURES)
ncols   = 3
nrows   = (n_feats + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4))
axes = axes.flatten()

for i, feat in enumerate(AUDIO_FEATURES):
    for lbl, col in zip(["High", "Low"], ["#2ecc71", "#e74c3c"]):
        sub = df[df["Popularity_Label"] == lbl][feat].dropna()
        axes[i].hist(sub, bins=30, color=col, alpha=0.45,
                     density=True, label=lbl)
        sub.plot.kde(ax=axes[i], color=col, linewidth=2)
    axes[i].set_title(f"{feat}")
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Audio Feature Distributions — High vs Low Popularity Tracks",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda/02_feature_distributions.png", bbox_inches="tight")
plt.show()
print("💡 Key Insight: Energy & Loudness are consistently higher in popular tracks.")
print("   Instrumentalness is the strongest separator — low in hits, high in obscure tracks.")

💡 Key Insight: Energy & Loudness are consistently higher in popular tracks.
   Instrumentalness is the strongest separator — low in hits, high in obscure tracks.


In [6]:
# ─────────────────────────────────────────────────────────────
# Phase 2.4 | Boxplots — High vs Low per Feature
# ─────────────────────────────────────────────────────────────
ncols = 3
nrows = (len(AUDIO_FEATURES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4))
axes = axes.flatten()

for i, feat in enumerate(AUDIO_FEATURES):
    sns.boxplot(data=df, x="Popularity_Label", y=feat,
                palette={"High": "#2ecc71", "Low": "#e74c3c"},
                ax=axes[i], width=0.5, fliersize=3)
    axes[i].set_title(f"{feat}")
    axes[i].set_xlabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Boxplots — Feature Values by Popularity Label",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda/03_boxplots.png", bbox_inches="tight")
plt.show()

In [7]:
# ─────────────────────────────────────────────────────────────
# Phase 2.5 | Correlation Heatmap
# ─────────────────────────────────────────────────────────────
corr_cols = AUDIO_FEATURES + ["track_popularity"]
corr      = df[[c for c in corr_cols if c in df.columns]].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(15, 12))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap=cmap, center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5, annot_kws={"size": 9})
ax.set_title("Feature Correlation Matrix — Spotify Audio Features",
             fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda/04_correlation_heatmap.png", bbox_inches="tight")
plt.show()

top_corr = (corr["track_popularity"]
            .drop("track_popularity")
            .abs()
            .sort_values(ascending=False))
print("\n🔗 Features most correlated with track_popularity:")
for f, v in top_corr.head(8).items():
    print(f"   {f:<25} | r = {v:.4f}")


🔗 Features most correlated with track_popularity:
   instrumentalness          | r = 0.2445
   acousticness              | r = 0.2145
   loudness                  | r = 0.1979
   energy                    | r = 0.1767
   danceability              | r = 0.1265
   valence                   | r = 0.0923
   tempo                     | r = 0.0573
   liveness                  | r = 0.0277


In [8]:
# ─────────────────────────────────────────────────────────────
# Phase 2.6 | Statistical Tests — T-Test & Mann-Whitney U
# ─────────────────────────────────────────────────────────────
df_lo = df[df["Popularity_Label"] == "Low"]
df_hi = df[df["Popularity_Label"] == "High"]

rows = []
for feat in AUDIO_FEATURES:
    if feat not in df.columns: continue
    a, b    = df_lo[feat].dropna(), df_hi[feat].dropna()
    t, p_t  = ttest_ind(a, b, equal_var=False)
    u, p_u  = mannwhitneyu(a, b, alternative="two-sided")
    d_mean  = b.mean() - a.mean()
    rows.append({"Feature": feat,
                 "Mean_Low": round(a.mean(), 4),
                 "Mean_High": round(b.mean(), 4),
                 "Delta": round(d_mean, 4),
                 "T-test p": round(p_t, 5),
                 "MWU p": round(p_u, 5),
                 "Significant": "✅" if p_t < 0.05 else "❌"})

stat_df = pd.DataFrame(rows).sort_values("T-test p")
print("📊 Statistical Tests — High vs Low Popularity Tracks\n")
display(stat_df)
stat_df.to_csv(OUTPUT_DIR / "eda/statistical_tests.csv", index=False)
sig_n = (stat_df["Significant"] == "✅").sum()
print(f"\n💡 {sig_n}/{len(stat_df)} features are statistically different (p < 0.05).")

📊 Statistical Tests — High vs Low Popularity Tracks



,Feature,Mean_Low,Mean_High,Delta,T-test p,MWU p,Significant
0,danceability,0.6058,0.6517,0.0459,0.00000,0.00000,✅
1,energy,0.5404,0.6627,0.1223,0.00000,0.00000,✅
2,loudness,-10.7475,-6.8068,3.9407,0.00000,0.00000,✅
4,acousticness,0.4094,0.2274,-0.1820,0.00000,0.00000,✅
7,valence,0.4561,0.5272,0.0711,0.00000,0.00000,✅
5,instrumentalness,0.2919,0.0425,-0.2494,0.00000,0.00000,✅
9,duration_ms,201605.6212,214022.3062,12416.6850,0.00000,0.00000,✅
8,tempo,116.8414,121.3253,4.4839,0.00000,0.00000,✅
10,time_signature,3.9280,3.9541,0.0260,0.02592,0.10900,✅
6,liveness,0.1654,0.1729,0.0075,0.06144,0.11439,❌



💡 9/13 features are statistically different (p < 0.05).


In [9]:
# ─────────────────────────────────────────────────────────────
# Phase 2.7 | Temporal & Genre Analysis
# ─────────────────────────────────────────────────────────────
if "track_album_release_date" in df.columns:
    df["track_album_release_date"] = pd.to_datetime(
        df["track_album_release_date"], errors="coerce")
    df["Release_Year"]       = df["track_album_release_date"].dt.year
    df["Release_Month"]      = df["track_album_release_date"].dt.month
    df["Release_DayOfWeek"]  = df["track_album_release_date"].dt.dayofweek

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    yr = df.groupby("Release_Year")["track_popularity"].mean()
    axes[0].plot(yr.index, yr.values, marker="o", color="#3498db",
                 linewidth=2.5, markersize=5)
    axes[0].fill_between(yr.index, yr.values, alpha=0.15, color="#3498db")
    axes[0].set_title("Mean Track Popularity by Release Year")
    axes[0].set_xlabel("Year"); axes[0].set_ylabel("Mean Popularity")

    yr_cnt = df.groupby("Release_Year").size()
    axes[1].bar(yr_cnt.index, yr_cnt.values, color="#9b59b6", alpha=0.8)
    axes[1].set_title("Track Count by Release Year")
    axes[1].set_xlabel("Year"); axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eda/05_temporal_analysis.png", bbox_inches="tight")
    plt.show()
    print("💡 Recent tracks (2020+) are disproportionately represented in the high-popularity set.")

if "playlist_genre" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # By genre
    gp = (df.groupby("playlist_genre")["track_popularity"]
            .agg(["mean","std"]).sort_values("mean", ascending=False))
    axes[0].bar(gp.index, gp["mean"], yerr=gp["std"],
                color=PALETTE[:len(gp)], capsize=5, alpha=0.85)
    axes[0].set_title("Mean Popularity by Playlist Genre")
    axes[0].set_xlabel("Genre"); axes[0].set_ylabel("Mean Popularity")
    axes[0].tick_params(axis="x", rotation=20)

    # Top subgenres
    if "playlist_subgenre" in df.columns:
        sp = (df.groupby("playlist_subgenre")["track_popularity"]
                .mean().nlargest(15))
        axes[1].barh(sp.index[::-1], sp.values[::-1], color="#e67e22", alpha=0.85)
        axes[1].set_title("Top 15 Subgenres by Mean Popularity")
        axes[1].set_xlabel("Mean Popularity")
    else:
        axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eda/06_genre_analysis.png", bbox_inches="tight")
    plt.show()

💡 Recent tracks (2020+) are disproportionately represented in the high-popularity set.


In [10]:
# ─────────────────────────────────────────────────────────────
# Phase 2.8 | Pairplot — Key Audio Features
# ─────────────────────────────────────────────────────────────
pair_feats = ["danceability", "energy", "loudness", "valence",
              "speechiness", "track_popularity"]
pair_feats = [c for c in pair_feats if c in df.columns]

pair_df = df[pair_feats + ["Popularity_Label"]].sample(
    min(1500, len(df)), random_state=SEED)

g = sns.pairplot(pair_df, hue="Popularity_Label",
                 palette={"High": "#2ecc71", "Low": "#e74c3c"},
                 plot_kws={"alpha": 0.35, "s": 20},
                 diag_kind="kde")
g.fig.suptitle("Pairplot — Key Audio Features", y=1.02, fontsize=14)
plt.savefig(OUTPUT_DIR / "eda/07_pairplot.png", bbox_inches="tight")
plt.show()

## 🛠️ Phase 3 — Advanced Feature Engineering

In [11]:
# ─────────────────────────────────────────────────────────────
# Phase 3.1 | Binary Target with Extreme-Quantile Strategy
# Mirrors original notebook logic; keeps only clearly Hit/NotHit
# ─────────────────────────────────────────────────────────────
df_fe = df.copy()

# Dataset-aware thresholds:
# high_popularity_data  → pop ≥ 68 (all) → all are positive class
# low_popularity_data   → pop ≤ 68 (all) → most are negative class
# We use quantile-based split on the merged data
HIGH_Q = df_fe["track_popularity"].quantile(0.80)
LOW_Q  = df_fe["track_popularity"].quantile(0.20)

df_fe["is_hit"] = (df_fe["track_popularity"] >= HIGH_Q).astype(int)

print(f"Hit threshold      : popularity ≥ {HIGH_Q:.1f}")
print(f"Not-hit threshold  : popularity ≤ {LOW_Q:.1f}")
print(f"Class distribution : {df_fe['is_hit'].value_counts().to_dict()}")
print(f"Class ratio        : {df_fe['is_hit'].mean():.2%} hits")

Hit threshold      : popularity ≥ 72.0
Not-hit threshold  : popularity ≤ 34.0
Class distribution : {0: 3488, 1: 1007}
Class ratio        : 22.40% hits


In [12]:
# ─────────────────────────────────────────────────────────────
# Phase 3.2 | Domain-Driven Feature Engineering
# ─────────────────────────────────────────────────────────────

# 1. Release date components
if "track_album_release_date" in df_fe.columns:
    df_fe["track_album_release_date"] = pd.to_datetime(
        df_fe["track_album_release_date"], errors="coerce")
    df_fe["Release_Year"]        = df_fe["track_album_release_date"].dt.year
    df_fe["Release_Month"]       = df_fe["track_album_release_date"].dt.month
    df_fe["Release_DayOfWeek"]   = df_fe["track_album_release_date"].dt.dayofweek
    df_fe["Track_Age_Days"]      = (
        pd.Timestamp("2025-01-01") - df_fe["track_album_release_date"]
    ).dt.days.clip(lower=0)

# 2. Audio interaction features
cols = df_fe.columns.tolist()
if "energy" in cols and "danceability" in cols:
    df_fe["energy_dance_product"] = df_fe["energy"] * df_fe["danceability"]
    df_fe["energy_dance_ratio"]   = df_fe["energy"] / (df_fe["danceability"] + 1e-6)
if "valence" in cols and "energy" in cols:
    df_fe["mood_score"]           = df_fe["valence"] * df_fe["energy"]
    df_fe["positivity_gap"]       = df_fe["valence"] - df_fe["energy"]
if "acousticness" in cols and "energy" in cols:
    df_fe["acoustic_energy_diff"] = df_fe["acousticness"] - df_fe["energy"]
if "speechiness" in cols and "instrumentalness" in cols:
    df_fe["vocal_score"]          = df_fe["speechiness"] - df_fe["instrumentalness"]
    df_fe["vocal_ratio"]          = df_fe["speechiness"] / (df_fe["instrumentalness"] + 1e-6)
if "loudness" in cols:
    df_fe["loudness_norm"]        = (df_fe["loudness"] + 60) / 60
if "tempo" in cols:
    df_fe["tempo_bin"] = pd.cut(
        df_fe["tempo"],
        bins=[0, 70, 100, 130, 160, 999],
        labels=["very_slow", "slow", "medium", "fast", "very_fast"])
if "duration_ms" in cols:
    df_fe["duration_min"]   = df_fe["duration_ms"] / 60_000
    df_fe["is_short_track"] = (df_fe["duration_min"] < 2.5).astype(int)
    df_fe["is_long_track"]  = (df_fe["duration_min"] > 5.0).astype(int)

# 3. Genre frequency encoding (how common is this genre?)
for cat_col in ["playlist_genre", "playlist_subgenre"]:
    if cat_col in df_fe.columns:
        freq = df_fe[cat_col].value_counts(normalize=True)
        df_fe[f"{cat_col}_freq"] = df_fe[cat_col].map(freq)

# 4. Artist hit-rate (target encoding with Bayesian smoothing)
if "track_artist" in df_fe.columns:
    global_rate  = df_fe["is_hit"].mean()
    K            = 20
    artist_stats = (df_fe.groupby("track_artist")["is_hit"]
                    .agg(["sum", "count"]).reset_index())
    artist_stats["artist_hit_rate"] = (
        (artist_stats["sum"] + K * global_rate) / (artist_stats["count"] + K))
    df_fe = df_fe.merge(
        artist_stats[["track_artist", "artist_hit_rate"]],
        on="track_artist", how="left")

print("✅  Feature engineering complete.")
print(f"   DataFrame shape: {df_fe.shape}")
new_feats = [c for c in df_fe.columns if c not in df.columns]
print(f"   New features ({len(new_feats)}): {new_feats}")

✅  Feature engineering complete.
   DataFrame shape: (4495, 45)
   New features (17): ['is_hit', 'Track_Age_Days', 'energy_dance_product', 'energy_dance_ratio', 'mood_score', 'positivity_gap', 'acoustic_energy_diff', 'vocal_score', 'vocal_ratio', 'loudness_norm', 'tempo_bin', 'duration_min', 'is_short_track', 'is_long_track', 'playlist_genre_freq', 'playlist_subgenre_freq', 'artist_hit_rate']


In [13]:
# ─────────────────────────────────────────────────────────────
# Phase 3.3 | Skewness Analysis & Log Transforms
# ─────────────────────────────────────────────────────────────
num_only = df_fe.select_dtypes(include=np.number).columns.tolist()
skew_s   = df_fe[num_only].skew().sort_values(ascending=False)
high_skew= skew_s[abs(skew_s) > 1.0].index.tolist()

log_created = []
for f in high_skew:
    if (df_fe[f] > 0).all():
        df_fe[f"log_{f}"] = np.log1p(df_fe[f])
        log_created.append(f)

print(f"Log-transformed {len(log_created)} features: {log_created}")

# Plot top skewed features
fig, ax = plt.subplots(figsize=(14, 4))
top_sk = skew_s.head(16)
clrs   = ["#e74c3c" if abs(v) > 1 else "#3498db" for v in top_sk.values]
ax.bar(top_sk.index, top_sk.values, color=clrs, alpha=0.85)
ax.axhline(1, color="#e74c3c", linestyle="--", linewidth=1, label="|skew|=1 threshold")
ax.axhline(-1, color="#e74c3c", linestyle="--", linewidth=1)
ax.set_title("Feature Skewness — Red bars were log-transformed")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "features/01_skewness.png", bbox_inches="tight")
plt.show()

Log-transformed 2 features: ['artist_hit_rate', 'playlist_subgenre_freq']


In [14]:
# ─────────────────────────────────────────────────────────────
# Phase 3.4 | Final Feature Matrix Assembly
# ─────────────────────────────────────────────────────────────
CORE_AUDIO = [c for c in [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_ms",
    "time_signature"
] if c in df_fe.columns]

ENGINEERED = [c for c in [
    "energy_dance_product", "energy_dance_ratio",
    "mood_score", "positivity_gap", "acoustic_energy_diff",
    "vocal_score", "vocal_ratio", "loudness_norm",
    "duration_min", "is_short_track", "is_long_track",
    "Release_Year", "Release_Month", "Release_DayOfWeek", "Track_Age_Days",
    "playlist_genre_freq", "playlist_subgenre_freq", "artist_hit_rate"
] if c in df_fe.columns]

LOG_FEATS = [c for c in df_fe.columns if c.startswith("log_")]

# One-hot encode low-cardinality categoricals
OHE_COLS = [c for c in ["playlist_genre", "playlist_subgenre", "mode", "tempo_bin"]
            if c in df_fe.columns and df_fe[c].nunique() <= 24]
if OHE_COLS:
    df_fe = pd.get_dummies(df_fe, columns=OHE_COLS, drop_first=True)
    OHE_FEATS = [c for c in df_fe.columns
                 if any(c.startswith(b + "_") for b in OHE_COLS)]
else:
    OHE_FEATS = []

# Key encoding
if "key" in df_fe.columns:
    df_fe = pd.get_dummies(df_fe, columns=["key"], prefix="key", drop_first=True)
    KEY_FEATS = [c for c in df_fe.columns if c.startswith("key_")]
else:
    KEY_FEATS = []

ALL_FEATURES = list(dict.fromkeys(
    CORE_AUDIO + ENGINEERED + LOG_FEATS + OHE_FEATS + KEY_FEATS
))
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_fe.columns]

# ── Build X, y ──────────────────────────────────────────────
X_full = df_fe[ALL_FEATURES].apply(pd.to_numeric, errors="coerce")
y_full = df_fe["is_hit"].astype(int)

# Drop rows with inf / NaN
valid  = X_full.replace([np.inf, -np.inf], np.nan).dropna().index
X_full = X_full.loc[valid]
y_full = y_full.loc[valid]

print(f"Feature matrix  : {X_full.shape}")
print(f"Target balance  : {y_full.value_counts().to_dict()}")
print(f"Features ({len(ALL_FEATURES)}): {ALL_FEATURES[:8]} …")

# ── Train / Test split ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, random_state=SEED, stratify=y_full)

# ── Robust scaling ───────────────────────────────────────────
scaler     = RobustScaler()
X_train_s  = pd.DataFrame(
    scaler.fit_transform(X_train), columns=ALL_FEATURES, index=X_train.index)
X_test_s   = pd.DataFrame(
    scaler.transform(X_test),      columns=ALL_FEATURES, index=X_test.index)

# ── SMOTE+Tomek resampling ───────────────────────────────────
smt          = SMOTETomek(random_state=SEED)
X_train_bal, y_train_bal = smt.fit_resample(X_train_s, y_train)
X_train_bal  = pd.DataFrame(X_train_bal, columns=ALL_FEATURES)
print(f"After SMOTETomek: {dict(zip(*np.unique(y_train_bal, return_counts=True)))}")

# Save feature importance placeholder
feature_importance_dict = {}
print("\n✅  Feature matrix ready for modeling.")

Feature matrix  : (4369, 47)
Target balance  : {0: 3409, 1: 960}
Features (47): ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence'] …
After SMOTETomek: {np.int64(0): np.int64(2724), np.int64(1): np.int64(2724)}

✅  Feature matrix ready for modeling.


## 🤖 Phase 4 — Baseline Model Framework (13 Classifiers)

In [15]:
# ─────────────────────────────────────────────────────────────
# Phase 4 | Baseline Models — All Standard Classifiers
# ─────────────────────────────────────────────────────────────
BASELINE_MODELS = {
    "Logistic Regression"    : LogisticRegression(
                                    max_iter=3000, C=1.0,
                                    class_weight="balanced", random_state=SEED),
    "Random Forest"          : RandomForestClassifier(
                                    n_estimators=400, max_depth=12,
                                    min_samples_split=10, min_samples_leaf=5,
                                    class_weight="balanced",
                                    random_state=SEED, n_jobs=-1),
    "Gradient Boosting"      : GradientBoostingClassifier(
                                    n_estimators=300, max_depth=5,
                                    learning_rate=0.05, random_state=SEED),
    "XGBoost"                : xgb.XGBClassifier(
                                    n_estimators=400, max_depth=6,
                                    learning_rate=0.05, subsample=0.8,
                                    colsample_bytree=0.8, eval_metric="logloss",
                                    random_state=SEED),
    "LightGBM"               : lgb.LGBMClassifier(
                                    n_estimators=400, max_depth=6,
                                    learning_rate=0.05, class_weight="balanced",
                                    random_state=SEED, verbose=-1),
    "CatBoost"               : cb.CatBoostClassifier(
                                    iterations=400, depth=6,
                                    learning_rate=0.05, verbose=0,
                                    random_seed=SEED),
    "Extra Trees"            : ExtraTreesClassifier(
                                    n_estimators=400,
                                    class_weight="balanced",
                                    random_state=SEED, n_jobs=-1),
    "AdaBoost"               : AdaBoostClassifier(
                                    n_estimators=200, learning_rate=0.05,
                                    random_state=SEED),
    "Bagging (DT)"           : BaggingClassifier(
                                    estimator=DecisionTreeClassifier(max_depth=8),
                                    n_estimators=200,
                                    random_state=SEED, n_jobs=-1),
    "SVM (RBF)"              : SVC(
                                    kernel="rbf", probability=True,
                                    class_weight="balanced",
                                    random_state=SEED, C=1.0),
    "KNN (k=15)"             : KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    "Gaussian NB"            : GaussianNB(),
    "LDA"                    : LinearDiscriminantAnalysis(),
}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Train model and return a comprehensive metrics dict."""
    t0    = time.time()
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        proba = model.decision_function(X_te)
        proba = (proba - proba.min()) / (proba.max() - proba.min() + 1e-9)
    else:
        proba = preds.astype(float)
    elapsed = time.time() - t0

    return {
        "Model"    : name,
        "Accuracy" : round(accuracy_score(y_te, preds), 4),
        "F1"       : round(f1_score(y_te, preds, zero_division=0), 4),
        "Precision": round(precision_score(y_te, preds, zero_division=0), 4),
        "Recall"   : round(recall_score(y_te, preds, zero_division=0), 4),
        "AUC_ROC"  : round(roc_auc_score(y_te, proba), 4),
        "MCC"      : round(matthews_corrcoef(y_te, preds), 4),
        "LogLoss"  : round(log_loss(y_te, proba), 4),
        "Brier"    : round(brier_score_loss(y_te, proba), 4),
        "Train_s"  : round(elapsed, 2),
        # Private keys (not written to CSV)
        "_model"   : model,
        "_preds"   : preds,
        "_proba"   : proba,
    }

print("🏃  Training 13 baseline classifiers …\n")
baseline_results = []
for name, mdl in BASELINE_MODELS.items():
    try:
        r = evaluate_model(name, mdl, X_train_bal, y_train_bal, X_test_s, y_test)
        baseline_results.append(r)
        print(f"  ✅  {name:<28}  Acc={r['Accuracy']:.4f}  "
              f"AUC={r['AUC_ROC']:.4f}  F1={r['F1']:.4f}  ({r['Train_s']}s)")
        # Save fitted models for feature importance
        feature_importance_dict[name] = mdl
    except Exception as e:
        print(f"  ⚠️  {name} skipped: {e}")

baseline_df = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith("_")}
     for r in baseline_results]
).sort_values("AUC_ROC", ascending=False).reset_index(drop=True)

print("\n📊  Baseline Leaderboard:")
display(baseline_df)
baseline_df.to_csv(OUTPUT_DIR / "models/baseline_leaderboard.csv", index=False)

🏃  Training 13 baseline classifiers …

  ✅  Logistic Regression           Acc=0.9634  AUC=0.9842  F1=0.9192  (0.07s)
  ✅  Random Forest                 Acc=0.9588  AUC=0.9922  F1=0.9095  (5.95s)
  ✅  Gradient Boosting             Acc=0.9588  AUC=0.9929  F1=0.9082  (34.44s)
  ✅  XGBoost                       Acc=0.9588  AUC=0.9894  F1=0.9077  (8.86s)
  ✅  LightGBM                      Acc=0.9542  AUC=0.9928  F1=0.8969  (2.58s)
  ✅  CatBoost                      Acc=0.9519  AUC=0.9922  F1=0.8918  (7.47s)
  ✅  Extra Trees                   Acc=0.9531  AUC=0.9860  F1=0.8957  (2.06s)
  ✅  AdaBoost                      Acc=0.9634  AUC=0.9944  F1=0.9204  (4.85s)
  ✅  Bagging (DT)                  Acc=0.9542  AUC=0.9927  F1=0.8985  (18.79s)
  ✅  SVM (RBF)                     Acc=0.9645  AUC=0.9855  F1=0.9242  (3.66s)
  ✅  KNN (k=15)                    Acc=0.9451  AUC=0.9782  F1=0.8852  (0.14s)
  ✅  Gaussian NB                   Acc=0.6373  AUC=0.8703  F1=0.5160  (0.01s)
  ✅  LDA               

,Model,Accuracy,F1,Precision,Recall,AUC_ROC,MCC,LogLoss,Brier,Train_s
0,AdaBoost,0.9634,0.9204,0.8810,0.9635,0.9944,0.8982,0.2276,0.0488,4.85
1,Gradient Boosting,0.9588,0.9082,0.8900,0.9271,0.9929,0.8819,0.1096,0.0321,34.44
2,LightGBM,0.9542,0.8969,0.8878,0.9062,0.9928,0.8676,0.1447,0.0365,2.58
3,Bagging (DT),0.9542,0.8985,0.8762,0.9219,0.9927,0.8694,0.0895,0.0298,18.79
4,Random Forest,0.9588,0.9095,0.8786,0.9427,0.9922,0.8838,0.1048,0.0284,5.95
5,CatBoost,0.9519,0.8918,0.8827,0.9010,0.9922,0.8610,0.1121,0.0341,7.47
6,XGBoost,0.9588,0.9077,0.8939,0.9219,0.9894,0.8814,0.1145,0.0317,8.86
7,Extra Trees,0.9531,0.8957,0.8756,0.9167,0.9860,0.8658,0.1503,0.0381,2.06
8,SVM (RBF),0.9645,0.9242,0.8710,0.9844,0.9855,0.9040,0.1289,0.0299,3.66
9,Logistic Regression,0.9634,0.9192,0.8922,0.9479,0.9842,0.8962,0.1704,0.0327,0.07


In [16]:
# ─────────────────────────────────────────────────────────────
# Phase 4.2 | Baseline Visual Comparison
# ─────────────────────────────────────────────────────────────
metrics = ["Accuracy", "F1", "AUC_ROC", "MCC"]
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    sub    = baseline_df.sort_values(metric, ascending=True)
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sub)))
    bars   = axes[i].barh(sub["Model"], sub[metric],
                           color=colors, edgecolor="white")
    axes[i].set_title(f"{metric}", fontsize=13)
    for bar, val in zip(bars, sub[metric]):
        axes[i].text(bar.get_width() + 0.003,
                     bar.get_y() + bar.get_height() / 2,
                     f"{val:.4f}", va="center", fontsize=9)
    axes[i].set_xlim(max(0, sub[metric].min() - 0.05), 1.02)

fig.suptitle("Baseline Model Comparison — 4 Metrics", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "models/01_baseline_comparison.png", bbox_inches="tight")
plt.show()

In [17]:
# ─────────────────────────────────────────────────────────────
# Phase 4.3 | ROC + Precision-Recall Curves
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
cmap = plt.cm.get_cmap("tab20", len(baseline_results))

for i, r in enumerate(sorted(baseline_results,
                               key=lambda x: x["AUC_ROC"], reverse=True)):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, r["_proba"])
    axes[0].plot(fpr, tpr, label=f"{r['Model']} ({r['AUC_ROC']:.3f})",
                 color=cmap(i), linewidth=1.5)
    # Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, r["_proba"])
    axes[1].plot(rec, prec, label=f"{r['Model']} (AP={average_precision_score(y_test, r['_proba']):.3f})",
                 color=cmap(i), linewidth=1.5)

axes[0].plot([0,1],[0,1],"k--", linewidth=1)
axes[0].set_title("ROC Curves — Baseline Models")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=7.5, loc="lower right")
axes[0].grid(alpha=0.3)

baseline_rate = y_test.mean()
axes[1].axhline(baseline_rate, color="k", linestyle="--",
                label=f"Baseline={baseline_rate:.3f}")
axes[1].set_title("Precision-Recall Curves — Baseline Models")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=7.5, loc="upper right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "models/02_roc_pr_baseline.png", bbox_inches="tight")
plt.show()

## 🚀 Phase 5 — Advanced Hyperparameter Tuning (Optuna + GPU)

In [18]:
# ─────────────────────────────────────────────────────────────
# Phase 5.1 | GPU Detection
# ─────────────────────────────────────────────────────────────
import subprocess as _sp

try:
    _gpu_out = _sp.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                  "--format=csv,noheader"], text=True).strip()
    print(f"🎮  GPU detected: {_gpu_out}")
    USE_GPU    = True
    XGB_DEVICE = "cuda"
    LGB_DEVICE = "gpu"
except Exception:
    print("💻  No GPU — running on CPU (will be slower)")
    USE_GPU    = False
    XGB_DEVICE = "cpu"
    LGB_DEVICE = "cpu"

SKF = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
N_TRIALS = 20   # Increase for better results; 60 is a good balance

🎮  GPU detected: Tesla T4, 15360 MiB


In [19]:
# ─────────────────────────────────────────────────────────────
# Phase 5.2 | Optuna — XGBoost Tuning
# ─────────────────────────────────────────────────────────────
def _xgb_objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int("n_estimators", 200, 1200, step=100),
        max_depth        = trial.suggest_int("max_depth", 3, 12),
        learning_rate    = trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        subsample        = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        colsample_bylevel= trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        min_child_weight = trial.suggest_int("min_child_weight", 1, 15),
        gamma            = trial.suggest_float("gamma", 0, 5),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        max_delta_step   = trial.suggest_int("max_delta_step", 0, 10),
        eval_metric="logloss", random_state=SEED,
    )
    if USE_GPU:
        p["device"] = "cuda"
    m = xgb.XGBClassifier(**p)
    return cross_val_score(m, X_train_bal, y_train_bal,
                            cv=SKF, scoring="roc_auc", n_jobs=-1).mean()

print(f"🔍  Tuning XGBoost — {N_TRIALS} trials …")
xgb_study = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(_xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f"   Best CV AUC : {xgb_study.best_value:.4f}")

🔍  Tuning XGBoost — 60 trials …


  0%|          | 0/60 [00:00<?, ?it/s]

   Best CV AUC : 0.9972


In [20]:
# ─────────────────────────────────────────────────────────────
# Phase 5.3 | Optuna — LightGBM Tuning
# ─────────────────────────────────────────────────────────────
def _lgb_objective(trial):
    p = dict(
        n_estimators      = trial.suggest_int("n_estimators", 200, 1200, step=100),
        max_depth         = trial.suggest_int("max_depth", 3, 12),
        learning_rate     = trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        num_leaves        = trial.suggest_int("num_leaves", 16, 256),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 80),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_split_gain    = trial.suggest_float("min_split_gain", 0, 1.0),
        class_weight="balanced", random_state=SEED, verbose=-1,
    )
    if USE_GPU:
        p["device"] = "gpu"
    m = lgb.LGBMClassifier(**p)
    return cross_val_score(m, X_train_bal, y_train_bal,
                            cv=SKF, scoring="roc_auc", n_jobs=-1).mean()

print(f"🔍  Tuning LightGBM — {N_TRIALS} trials …")
lgb_study = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(_lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f"   Best CV AUC : {lgb_study.best_value:.4f}")

🔍  Tuning LightGBM — 60 trials …


  0%|          | 0/60 [00:00<?, ?it/s]

   Best CV AUC : 0.9977


In [21]:
# ─────────────────────────────────────────────────────────────
# Phase 5.4 | Optuna — Lightweight Random Forest Tuning
# ─────────────────────────────────────────────────────────────
def _rf_objective(trial):
    p = dict(
        n_estimators      = trial.suggest_int("n_estimators", 100, 400, step=100),
        max_depth         = trial.suggest_int("max_depth", 5, 15),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features      = trial.suggest_categorical(
                                "max_features",
                                ["sqrt", "log2"]
                            ),
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )

    m = RandomForestClassifier(**p)

    return cross_val_score(
        m,
        X_train_bal,
        y_train_bal,
        cv=SKF,
        scoring="roc_auc",
        n_jobs=-1
    ).mean()

print(f"🔍 Tuning Random Forest — {N_TRIALS} trials ...")

rf_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

rf_study.optimize(
    _rf_objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f"Best CV AUC: {rf_study.best_value:.4f}")

🔍 Tuning Random Forest — 60 trials ...


  0%|          | 0/60 [00:00<?, ?it/s]

Best CV AUC: 0.9966


In [24]:
# ─────────────────────────────────────────────────────────────
# Phase 5.5 | Optuna — Lightweight CatBoost Tuning
# ─────────────────────────────────────────────────────────────
def _cb_objective(trial):
    p = dict(
        iterations      = trial.suggest_int("iterations", 200, 600, step=100),
        depth           = trial.suggest_int("depth", 4, 8),
        learning_rate   = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        l2_leaf_reg     = trial.suggest_float("l2_leaf_reg", 1, 5),
        verbose         = 0,
        random_seed     = SEED,
    )

    m = cb.CatBoostClassifier(**p)

    return cross_val_score(
        m,
        X_train_bal,
        y_train_bal,
        cv=SKF,
        scoring="roc_auc",
        n_jobs=-1
    ).mean()

print(f"🔍 Tuning CatBoost — {N_TRIALS} trials ...")

cb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

cb_study.optimize(
    _cb_objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print(f"Best CV AUC: {cb_study.best_value:.4f}")

🔍 Tuning CatBoost — 60 trials ...


  0%|          | 0/60 [00:00<?, ?it/s]

Best CV AUC: 0.9967


In [25]:
# ─────────────────────────────────────────────────────────────
# Phase 5.6 | Train All Tuned Models
# ─────────────────────────────────────────────────────────────
def _build_tuned():
    xp = {**xgb_study.best_params,
          "eval_metric": "logloss", "random_state": SEED}
    if USE_GPU: xp["device"] = "cuda"

    lp = {**lgb_study.best_params,
          "class_weight": "balanced", "random_state": SEED, "verbose": -1}
    if USE_GPU: lp["device"] = "gpu"

    rp = {**rf_study.best_params,
          "class_weight": "balanced", "random_state": SEED, "n_jobs": -1}

    cp = {**cb_study.best_params, "verbose": 0, "random_seed": SEED}

    return {
        "XGBoost (Tuned)"     : xgb.XGBClassifier(**xp),
        "LightGBM (Tuned)"    : lgb.LGBMClassifier(**lp),
        "RandomForest (Tuned)": RandomForestClassifier(**rp),
        "CatBoost (Tuned)"    : cb.CatBoostClassifier(**cp),
    }

TUNED_MODELS = _build_tuned()

print("🏋️  Training tuned models …\n")
tuned_results = []
for name, mdl in TUNED_MODELS.items():
    try:
        r = evaluate_model(name, mdl, X_train_bal, y_train_bal, X_test_s, y_test)
        tuned_results.append(r)
        print(f"  ✅  {name:<32}  Acc={r['Accuracy']:.4f}  "
              f"AUC={r['AUC_ROC']:.4f}  F1={r['F1']:.4f}")
        feature_importance_dict[name] = mdl
    except Exception as e:
        print(f"  ⚠️  {name} skipped: {e}")

tuned_df = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith("_")}
     for r in tuned_results]
).sort_values("AUC_ROC", ascending=False)
display(tuned_df)
tuned_df.to_csv(OUTPUT_DIR / "models/tuned_leaderboard.csv", index=False)

🏋️  Training tuned models …

  ✅  XGBoost (Tuned)                   Acc=0.9600  AUC=0.9930  F1=0.9109
  ✅  LightGBM (Tuned)                  Acc=0.9588  AUC=0.9931  F1=0.9077
  ✅  RandomForest (Tuned)              Acc=0.9565  AUC=0.9901  F1=0.9036
  ✅  CatBoost (Tuned)                  Acc=0.9531  AUC=0.9920  F1=0.8946


,Model,Accuracy,F1,Precision,Recall,AUC_ROC,MCC,LogLoss,Brier,Train_s
1,LightGBM (Tuned),0.9588,0.9077,0.8939,0.9219,0.9931,0.8814,0.1129,0.0330,1.59
0,XGBoost (Tuned),0.9600,0.9109,0.8905,0.9323,0.9930,0.8855,0.0930,0.0283,1.33
3,CatBoost (Tuned),0.9531,0.8946,0.8832,0.9062,0.9920,0.8646,0.1302,0.0362,9.81
2,RandomForest (Tuned),0.9565,0.9036,0.8812,0.9271,0.9901,0.8760,0.1112,0.0306,3.56


## 🏗️ Phase 6 — Ensemble & Stacking

In [26]:
# ─────────────────────────────────────────────────────────────
# Phase 6 | Voting & Stacking Ensembles
# ─────────────────────────────────────────────────────────────
def _get(name_substr):
    """Get the fitted model from tuned_results by name substring."""
    for r in tuned_results:
        if name_substr in r["Model"]:
            return r["_model"]
    for r in baseline_results:
        if name_substr in r["Model"]:
            return r["_model"]
    return None

_xgb = _get("XGBoost")
_lgb = _get("LightGBM")
_rf  = _get("RandomForest")
_cb  = _get("CatBoost")
_et  = _get("Extra Trees")

# ── Soft Voting ─────────────────────────────────────────────
vclf = VotingClassifier(
    estimators=[e for e in [
        ("xgb", _xgb), ("lgb", _lgb), ("rf", _rf), ("cb", _cb)
    ] if e[1] is not None],
    voting="soft", n_jobs=-1)
r_vote = evaluate_model("Soft Voting Ensemble",
                         vclf, X_train_bal, y_train_bal, X_test_s, y_test)
print(f"🗳️   Voting Ensemble   AUC={r_vote['AUC_ROC']:.4f}  F1={r_vote['F1']:.4f}")

# ── Stacking ────────────────────────────────────────────────
xp2 = {**xgb_study.best_params, "eval_metric": "logloss", "random_state": SEED}
if USE_GPU: xp2["device"] = "cuda"
lp2 = {**lgb_study.best_params, "class_weight": "balanced", "random_state": SEED, "verbose": -1}
if USE_GPU: lp2["device"] = "gpu"
rp2 = {**rf_study.best_params, "class_weight": "balanced", "random_state": SEED, "n_jobs": -1}
cp2 = {**cb_study.best_params, "verbose": 0, "random_seed": SEED}

sclf = StackingClassifier(
    estimators=[
        ("xgb", xgb.XGBClassifier(**xp2)),
        ("lgb", lgb.LGBMClassifier(**lp2)),
        ("rf",  RandomForestClassifier(**rp2)),
        ("cb",  cb.CatBoostClassifier(**cp2)),
        ("et",  ExtraTreesClassifier(n_estimators=400,
                                     class_weight="balanced",
                                     random_state=SEED, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(max_iter=2000, C=0.5,
                                        class_weight="balanced",
                                        random_state=SEED),
    cv=5, n_jobs=-1, passthrough=False)

r_stack = evaluate_model("Stacking Ensemble",
                          sclf, X_train_bal, y_train_bal, X_test_s, y_test)
print(f"🏗️   Stacking Ensemble  AUC={r_stack['AUC_ROC']:.4f}  F1={r_stack['F1']:.4f}")

ensemble_results = [r_vote, r_stack]

🗳️   Voting Ensemble   AUC=0.9925  F1=0.9003
🏗️   Stacking Ensemble  AUC=0.9896  F1=0.8964


## 📊 Phase 7 — Full Model Comparison with Visuals

In [27]:
# ─────────────────────────────────────────────────────────────
# Phase 7.1 | Master Leaderboard
# ─────────────────────────────────────────────────────────────
all_results = baseline_results + tuned_results + ensemble_results

master_df = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith("_")}
     for r in all_results]
).sort_values("AUC_ROC", ascending=False).reset_index(drop=True)
master_df.insert(0, "Rank", range(1, len(master_df) + 1))

# Tag model tier
def _tag(name):
    if "Tuned" in name or "Stacking" in name or "Voting" in name: return "Advanced"
    if name in ["XGBoost","LightGBM","CatBoost","Gradient Boosting"]: return "Boosting"
    if name in ["Random Forest","Extra Trees","AdaBoost","Bagging (DT)"]: return "Ensemble"
    return "Baseline"
master_df["Tier"] = master_df["Model"].apply(_tag)

master_df.to_csv(OUTPUT_DIR / "comparison/master_leaderboard.csv", index=False)
print("🏆  Master Leaderboard:")
display(master_df[["Rank","Model","Tier","Accuracy","F1","Precision","Recall","AUC_ROC","MCC","LogLoss","Brier"]])

🏆  Master Leaderboard:


,Rank,Model,Tier,Accuracy,F1,Precision,Recall,AUC_ROC,MCC,LogLoss,Brier
0,1,AdaBoost,Ensemble,0.9634,0.9204,0.8810,0.9635,0.9944,0.8982,0.2276,0.0488
1,2,LightGBM (Tuned),Advanced,0.9588,0.9077,0.8939,0.9219,0.9931,0.8814,0.1129,0.0330
2,3,XGBoost (Tuned),Advanced,0.9600,0.9109,0.8905,0.9323,0.9930,0.8855,0.0930,0.0283
3,4,Gradient Boosting,Boosting,0.9588,0.9082,0.8900,0.9271,0.9929,0.8819,0.1096,0.0321
4,5,LightGBM,Boosting,0.9542,0.8969,0.8878,0.9062,0.9928,0.8676,0.1447,0.0365
5,6,Bagging (DT),Ensemble,0.9542,0.8985,0.8762,0.9219,0.9927,0.8694,0.0895,0.0298
6,7,Soft Voting Ensemble,Advanced,0.9554,0.9003,0.8844,0.9167,0.9925,0.8718,0.0997,0.0305
7,8,CatBoost,Boosting,0.9519,0.8918,0.8827,0.9010,0.9922,0.8610,0.1121,0.0341
8,9,Random Forest,Ensemble,0.9588,0.9095,0.8786,0.9427,0.9922,0.8838,0.1048,0.0284
9,10,CatBoost (Tuned),Advanced,0.9531,0.8946,0.8832,0.9062,0.9920,0.8646,0.1302,0.0362


In [28]:
# ─────────────────────────────────────────────────────────────
# Phase 7.2 | Heatmap Leaderboard (All Metrics)
# ─────────────────────────────────────────────────────────────
metric_cols = ["Accuracy","F1","Precision","Recall","AUC_ROC","MCC"]
heat_data   = master_df.set_index("Model")[metric_cols]

fig, ax = plt.subplots(figsize=(14, max(8, len(heat_data) * 0.55)))
sns.heatmap(heat_data, annot=True, fmt=".4f", cmap="YlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Score"})
ax.set_title("Model Performance Heatmap — All Metrics", fontsize=14, pad=12)
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison/01_heatmap_leaderboard.png", bbox_inches="tight")
plt.show()

In [29]:
# ─────────────────────────────────────────────────────────────
# Phase 7.3 | Grouped Bar Chart — Top-10 Models
# ─────────────────────────────────────────────────────────────
top_n    = min(12, len(master_df))
top_mods = master_df.head(top_n)
metrics  = ["Accuracy", "F1", "AUC_ROC", "MCC"]
x        = np.arange(len(top_mods))
w        = 0.19

fig, ax = plt.subplots(figsize=(22, 7))
for i, metric in enumerate(metrics):
    bars = ax.bar(x + i * w, top_mods[metric], w,
                  label=metric, alpha=0.88, edgecolor="white")
    for bar, val in zip(bars, top_mods[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7, rotation=90)

ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(top_mods["Model"], rotation=38, ha="right", fontsize=9)
ax.set_ylim(0.45, 1.10)
ax.set_ylabel("Score")
ax.set_title(f"Top-{top_n} Models — Multi-Metric Comparison", fontsize=14)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison/02_grouped_bar.png", bbox_inches="tight")
plt.show()

In [30]:
# ─────────────────────────────────────────────────────────────
# Phase 7.4 | Confusion Matrices — Top 6 Models
# ─────────────────────────────────────────────────────────────
top6_names = master_df.head(6)["Model"].tolist()
top6       = [r for r in all_results if r["Model"] in top6_names]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, r in enumerate(top6):
    cm   = confusion_matrix(y_test, r["_preds"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Hit", "Hit"])
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    tier = master_df.loc[master_df["Model"] == r["Model"], "Tier"].values[0]
    axes[i].set_title(
        f"{r['Model']}  [{tier}]\n"
        f"Acc={r['Accuracy']:.4f}  AUC={r['AUC_ROC']:.4f}  F1={r['F1']:.4f}",
        fontsize=10)

plt.suptitle("Confusion Matrices — Top 6 Models", fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison/03_confusion_matrices.png", bbox_inches="tight")
plt.show()

In [31]:
# ─────────────────────────────────────────────────────────────
# Phase 7.5 | ROC + PR Curves — All Models
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
cmap = plt.cm.get_cmap("tab20", len(all_results))

for i, r in enumerate(sorted(all_results, key=lambda x: x["AUC_ROC"], reverse=True)):
    is_adv = "Tuned" in r["Model"] or "Ensemble" in r["Model"]
    lw     = 2.5 if is_adv else 1.2
    ls     = "-"  if is_adv else "--"

    fpr, tpr, _ = roc_curve(y_test, r["_proba"])
    axes[0].plot(fpr, tpr, color=cmap(i), linewidth=lw, linestyle=ls,
                 label=f"{r['Model']} ({r['AUC_ROC']:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, r["_proba"])
    ap = average_precision_score(y_test, r["_proba"])
    axes[1].plot(rec, prec, color=cmap(i), linewidth=lw, linestyle=ls,
                 label=f"{r['Model']} (AP={ap:.3f})")

axes[0].plot([0,1],[0,1],"k--", linewidth=1)
axes[0].set_title("ROC Curves — All Models  (solid = tuned/ensemble)")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=7, loc="lower right")
axes[0].grid(alpha=0.25)

axes[1].axhline(y_test.mean(), color="k", linestyle="--",
                label=f"No-skill = {y_test.mean():.3f}")
axes[1].set_title("Precision-Recall Curves — All Models")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=7, loc="upper right")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison/04_roc_pr_all.png", bbox_inches="tight")
plt.show()

In [32]:
# ─────────────────────────────────────────────────────────────
# Phase 7.6 | Calibration Curves (Top 4)
# ─────────────────────────────────────────────────────────────
top4_res = sorted(all_results, key=lambda x: x["AUC_ROC"], reverse=True)[:4]

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0,1],[0,1],"k--", linewidth=1, label="Perfect calibration")

for r in top4_res:
    frac_pos, mean_pred = calibration_curve(y_test, r["_proba"], n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", linewidth=2,
            label=f"{r['Model']}")

ax.set_title("Calibration Curves — Top 4 Models", fontsize=13)
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison/05_calibration.png", bbox_inches="tight")
plt.show()
print("💡 A well-calibrated model follows the dashed diagonal.")

💡 A well-calibrated model follows the dashed diagonal.


## 🔬 Phase 8 — Explainability: SHAP, Permutation & Feature Importance

In [33]:
# ─────────────────────────────────────────────────────────────
# Phase 8.1 | Built-in Feature Importance (Tree Models)
# ─────────────────────────────────────────────────────────────
tree_models = {
    name: r["_model"] for r in (tuned_results + baseline_results)
    if hasattr(r["_model"], "feature_importances_")
    for name in [r["Model"]]
}

n_tm  = len(tree_models)
ncols = 2
nrows = (n_tm + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 5))
axes = axes.flatten()

for i, (name, mdl) in enumerate(list(tree_models.items())[:nrows * ncols]):
    imp = pd.Series(mdl.feature_importances_, index=ALL_FEATURES)
    top = imp.nlargest(15).sort_values()
    axes[i].barh(top.index, top.values, color=PALETTE[i % len(PALETTE)], alpha=0.85)
    axes[i].set_title(f"Feature Importance — {name}", fontsize=11)
    axes[i].set_xlabel("Importance Score")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Built-in Feature Importance — All Tree Models", fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xai/01_builtin_importance.png", bbox_inches="tight")
plt.show()

In [34]:
# ─────────────────────────────────────────────────────────────
# Phase 8.2 | SHAP Summary & Bar — XGBoost (Best Tree)
# ─────────────────────────────────────────────────────────────
_xgb_tuned = next((r["_model"] for r in tuned_results
                    if "XGBoost" in r["Model"]), None)
if _xgb_tuned is None:
    _xgb_tuned = next(r["_model"] for r in baseline_results
                       if "XGBoost" in r["Model"])

print("⚡  Computing SHAP values for XGBoost (Tuned) …")
explainer = shap.TreeExplainer(_xgb_tuned)
shap_vals = explainer.shap_values(X_test_s)

# Beeswarm / summary
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals, X_test_s, feature_names=ALL_FEATURES,
                   show=False, plot_size=None)
plt.title("SHAP Summary (Beeswarm) — XGBoost Tuned", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xai/02_shap_beeswarm.png", bbox_inches="tight")
plt.show()
print("💡 Red dots push toward 'Hit'; blue dots push toward 'Not Hit'.")
print("   Features are ordered by mean absolute SHAP value (top = most important).")

⚡  Computing SHAP values for XGBoost (Tuned) …
💡 Red dots push toward 'Hit'; blue dots push toward 'Not Hit'.
   Features are ordered by mean absolute SHAP value (top = most important).


In [35]:
# ─────────────────────────────────────────────────────────────
# Phase 8.3 | SHAP Bar Plot + Interaction Dot
# ─────────────────────────────────────────────────────────────
# Bar (global importance)
plt.figure(figsize=(11, 7))
shap.summary_plot(shap_vals, X_test_s, feature_names=ALL_FEATURES,
                   plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean |SHAP|) — XGBoost Tuned", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xai/03_shap_bar.png", bbox_inches="tight")
plt.show()

In [36]:
# ─────────────────────────────────────────────────────────────
# Phase 8.4 | SHAP Dependence Plots — Top 4 Features
# ─────────────────────────────────────────────────────────────
mean_abs  = np.abs(shap_vals).mean(axis=0)
top4_idx  = np.argsort(mean_abs)[::-1][:4]
top4_feats= [ALL_FEATURES[i] for i in top4_idx]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, feat in zip(axes.flatten(), top4_feats):
    fidx = ALL_FEATURES.index(feat)
    shap.dependence_plot(fidx, shap_vals, X_test_s.values,
                          feature_names=ALL_FEATURES, ax=ax, show=False,
                          interaction_index="auto")
    ax.set_title(f"SHAP Dependence: {feat}")

plt.suptitle("SHAP Dependence Plots — Top 4 Features", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xai/04_shap_dependence.png", bbox_inches="tight")
plt.show()
print("💡 Each dot = one track. X-axis = feature value, Y-axis = impact on prediction.")
print("   Color = value of the interacting feature (auto-selected).")

💡 Each dot = one track. X-axis = feature value, Y-axis = impact on prediction.
   Color = value of the interacting feature (auto-selected).


In [37]:
# ─────────────────────────────────────────────────────────────
# Phase 8.5 | SHAP Waterfall — Individual Predictions
# ─────────────────────────────────────────────────────────────
_xgb_preds = _xgb_tuned.predict(X_test_s)
y_arr       = y_test.values

# Find a correct hit and a missed hit
correct_hit_idx = np.where((y_arr == 1) & (_xgb_preds == 1))[0]
missed_hit_idx  = np.where((y_arr == 1) & (_xgb_preds == 0))[0]
false_pos_idx   = np.where((y_arr == 0) & (_xgb_preds == 1))[0]

exp = shap.Explanation(
    values       = shap_vals,
    base_values  = explainer.expected_value,
    data         = X_test_s.values,
    feature_names= ALL_FEATURES
)

fig, axes = plt.subplots(1, 3, figsize=(24, 6))
titles_idx = [
    ("✅ Correct HIT",          correct_hit_idx[0]  if len(correct_hit_idx) else 0),
    ("❌ Missed HIT (FN)",      missed_hit_idx[0]   if len(missed_hit_idx)  else 1),
    ("⚠️ False Positive (FP)", false_pos_idx[0]    if len(false_pos_idx)   else 2),
]
for ax, (title, idx) in zip(axes, titles_idx):
    plt.sca(ax)
    shap.waterfall_plot(exp[idx], max_display=10, show=False)
    ax.set_title(title, fontsize=11)

plt.suptitle("SHAP Waterfall Plots — 3 Prediction Types", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xai/05_shap_waterfall.png", bbox_inches="tight")
plt.show()

In [38]:
# ─────────────────────────────────────────────────────────────
# Phase 8.6 | Permutation Importance (Model-Agnostic)
# ─────────────────────────────────────────────────────────────
_lgb_tuned = next((r["_model"] for r in tuned_results
                    if "LightGBM" in r["Model"]), None)
if _lgb_tuned is None:
    _lgb_tuned = next(r["_model"] for r in baseline_results
                       if "LightGBM" in r["Model"])

print("⚡  Computing permutation importance for LightGBM (Tuned) …")
perm = permutation_importance(
    _lgb_tuned, X_test_s, y_test,
    n_repeats=20, random_state=SEED, n_jobs=-1,
    scoring="roc_auc")

perm_df = pd.DataFrame({
    "Feature"   : ALL_FEATURES,
    "Importance": perm.importances_mean,
    "Std"       : perm.importances_std,
}).sort_values("Importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(perm_df["Feature"][::-1], perm_df["Importance"][::-1],
        xerr=perm_df["Std"][::-1], capsize=4, color="#e67e22", alpha=0.85)
ax.set_title("Permutation Feature Importance (AUC-ROC) — LightGBM Tuned",
             fontsize=13)
ax.set_xlabel("Mean AUC-ROC Decrease when Feature is Permuted")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xai/06_permutation_importance.png", bbox_inches="tight")
plt.show()

perm_df.to_csv(OUTPUT_DIR / "xai/permutation_importance.csv", index=False)
print("💡 Larger bars = feature is more critical to model performance.")
print("   Error bars show variance across the 20 permutation rounds.")

⚡  Computing permutation importance for LightGBM (Tuned) …
💡 Larger bars = feature is more critical to model performance.
   Error bars show variance across the 20 permutation rounds.


In [39]:
# ─────────────────────────────────────────────────────────────
# Phase 8.7 | SHAP Force Plot (HTML artefact)
# ─────────────────────────────────────────────────────────────
from IPython.display import display as _disp, HTML as _HTML

shap.initjs()
_idx = correct_hit_idx[0] if len(correct_hit_idx) else 0
force_html = shap.force_plot(
    explainer.expected_value,
    shap_vals[_idx],
    X_test_s.iloc[_idx],
    feature_names=ALL_FEATURES,
    show=False,
    matplotlib=False
)
# Save as HTML
force_path = OUTPUT_DIR / "xai/shap_force_plot.html"
shap.save_html(str(force_path), force_html)
print(f"✅  Interactive SHAP force plot saved → {force_path}")
_disp(force_html)

✅  Interactive SHAP force plot saved → spotify_outputs/xai/shap_force_plot.html


## 💾 Phase 9 — Save All Outputs & Download ZIP

In [40]:
# ─────────────────────────────────────────────────────────────
# Phase 9.1 | Persist Models, Scalers, Features
# ─────────────────────────────────────────────────────────────

# Best model
best_name = master_df.iloc[0]["Model"]
best_r    = next(r for r in all_results if r["Model"] == best_name)

with open(OUTPUT_DIR / "models/best_model.pkl",  "wb") as f:
    pickle.dump(best_r["_model"], f)
with open(OUTPUT_DIR / "models/scaler.pkl",      "wb") as f:
    pickle.dump(scaler, f)
with open(OUTPUT_DIR / "models/features.json",   "w") as f:
    json.dump(ALL_FEATURES, f, indent=2)

print(f"✅  Best model '{best_name}' saved.")

# ── Classification reports ────────────────────────────────────
for r in all_results:
    safe_name = r["Model"].replace(" ", "_").replace("(", "").replace(")", "")
    report    = classification_report(y_test, r["_preds"],
                                       target_names=["Not Hit", "Hit"])
    path      = OUTPUT_DIR / f"reports/{safe_name}_report.txt"
    path.write_text(
        f"Model: {r['Model']}\n"
        f"AUC-ROC: {r['AUC_ROC']:.4f}  |  F1: {r['F1']:.4f}\n"
        f"{'─'*60}\n{report}\n"
    )

# ── SHAP values ────────────────────────────────────────────────
np.save(OUTPUT_DIR / "xai/shap_values.npy", shap_vals)
pd.DataFrame(shap_vals, columns=ALL_FEATURES).to_csv(
    OUTPUT_DIR / "xai/shap_values.csv", index=False)

# ── Optuna study params ────────────────────────────────────────
for study_name, study in [("xgboost", xgb_study), ("lightgbm", lgb_study),
                            ("randomforest", rf_study), ("catboost", cb_study)]:
    with open(OUTPUT_DIR / f"models/optuna_{study_name}_best_params.json", "w") as f:
        json.dump(study.best_params, f, indent=2)

print("✅  All artefacts saved.")

✅  Best model 'AdaBoost' saved.
✅  All artefacts saved.


In [41]:
# ─────────────────────────────────────────────────────────────
# Phase 9.2 | ZIP Everything & Download
# ─────────────────────────────────────────────────────────────
ZIP_NAME = f"spotify_ml_outputs_{TIMESTAMP}.zip"

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(OUTPUT_DIR.rglob("*")):
        if fpath.is_file():
            zf.write(fpath, fpath.relative_to(OUTPUT_DIR.parent))

size_mb = Path(ZIP_NAME).stat().st_size / 1e6
print(f"\n📦  ZIP created: {ZIP_NAME}  ({size_mb:.1f} MB)")
print(f"    Contents:")
with zipfile.ZipFile(ZIP_NAME) as zf:
    for name in sorted(zf.namelist()):
        print(f"      {name}")

from google.colab import files as _colab_files
_colab_files.download(ZIP_NAME)
print("\n✅  Download triggered.")


📦  ZIP created: spotify_ml_outputs_20260608_042631.zip  (6.2 MB)
    Contents:
      spotify_outputs/comparison/01_heatmap_leaderboard.png
      spotify_outputs/comparison/02_grouped_bar.png
      spotify_outputs/comparison/03_confusion_matrices.png
      spotify_outputs/comparison/04_roc_pr_all.png
      spotify_outputs/comparison/05_calibration.png
      spotify_outputs/comparison/master_leaderboard.csv
      spotify_outputs/eda/01_popularity_distribution.png
      spotify_outputs/eda/02_feature_distributions.png
      spotify_outputs/eda/03_boxplots.png
      spotify_outputs/eda/04_correlation_heatmap.png
      spotify_outputs/eda/05_temporal_analysis.png
      spotify_outputs/eda/06_genre_analysis.png
      spotify_outputs/eda/07_pairplot.png
      spotify_outputs/eda/statistical_tests.csv
      spotify_outputs/features/01_skewness.png
      spotify_outputs/models/01_baseline_comparison.png
      spotify_outputs/models/02_roc_pr_baseline.png
      spotify_outputs/models/baseline_l

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅  Download triggered.


In [42]:
# ─────────────────────────────────────────────────────────────
# Phase 9.3 | Final Summary Dashboard
# ─────────────────────────────────────────────────────────────
print()
print("╔" + "═"*63 + "╗")
print("║   🎵  SPOTIFY HIT PREDICTION — PIPELINE COMPLETE            ║")
print("╠" + "═"*63 + "╣")
print(f"║  Dataset rows analysed   : {len(X_full):>8,}                        ║")
print(f"║  Features engineered     : {len(ALL_FEATURES):>8}                        ║")
print(f"║  Total models trained    : {len(all_results):>8}                        ║")
print(f"║  Tuning trials (Optuna)  : {N_TRIALS * 4:>8} ({N_TRIALS} × 4 models)           ║")
print("╠" + "═"*63 + "╣")
print(f"║  🏆 Best Model  : {master_df.iloc[0]['Model']:<42}║")
print(f"║     Accuracy   : {master_df.iloc[0]['Accuracy']:.4f}                                     ║")
print(f"║     AUC-ROC    : {master_df.iloc[0]['AUC_ROC']:.4f}                                     ║")
print(f"║     F1-Score   : {master_df.iloc[0]['F1']:.4f}                                     ║")
print(f"║     MCC        : {master_df.iloc[0]['MCC']:.4f}                                     ║")
print("╠" + "═"*63 + "╣")
print(f"║  Output directory : {str(OUTPUT_DIR):<41}║")
print(f"║  ZIP downloaded   : {ZIP_NAME:<41}║")
print("╚" + "═"*63 + "╝")


╔═══════════════════════════════════════════════════════════════╗
║   🎵  SPOTIFY HIT PREDICTION — PIPELINE COMPLETE            ║
╠═══════════════════════════════════════════════════════════════╣
║  Dataset rows analysed   :    4,369                        ║
║  Features engineered     :       47                        ║
║  Total models trained    :       19                        ║
║  Tuning trials (Optuna)  :      240 (60 × 4 models)           ║
╠═══════════════════════════════════════════════════════════════╣
║  🏆 Best Model  : AdaBoost                                  ║
║     Accuracy   : 0.9634                                     ║
║     AUC-ROC    : 0.9944                                     ║
║     F1-Score   : 0.9204                                     ║
║     MCC        : 0.8982                                     ║
╠═══════════════════════════════════════════════════════════════╣
║  Output directory : spotify_outputs                          ║
║  ZIP downloaded   : spotify_ml_